In [60]:
from openfhe import *
import openfhe_numpy as onp
import jax.numpy as jnp
import numpy as np

In [120]:
mult_depth = 10

params = CCParamsCKKSRNS()
params.SetMultiplicativeDepth(mult_depth)
params.SetScalingModSize(59)
params.SetFirstModSize(60)
params.SetScalingTechnique(FIXEDAUTO)
params.SetKeySwitchTechnique(HYBRID)
params.SetSecretKeyDist(UNIFORM_TERNARY)
# params.SetRingDim(2*65536)

cc = GenCryptoContext(params)
cc.Enable(PKESchemeFeature.PKE)
cc.Enable(PKESchemeFeature.LEVELEDSHE)
cc.Enable(PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()

cc.EvalMultKeyGen(keys.secretKey)
cc.EvalSumKeyGen(keys.secretKey)

batch_size = cc.GetRingDimension() // 2
batch_size

32768

In [121]:
def make_compatible(enc_vec, keys, debug=False):
    length = enc_vec.original_shape[0]

    selection_matrix = np.eye(length)
    
    if debug:
        breakpoint()

    print(selection_matrix.shape)

    enc_selection = onp.array(
        cc=enc_vec.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_vec.batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey) # maybe this can be moved outside so only public key would be needed in this function
    # enc_selection.extra["rowkey"] = onp.sum_row_keys(keys.secretKey, enc_selection.ncols, enc_selection.batch_size) # needed for row sum

    slice_result = enc_selection @ enc_vec
    # import pdb; pdb.set_trace()
    return slice_result

In [123]:
# test matrix
x_in = np.random.rand(129)
x = jnp.array(x_in)

x_plain = onp.array(
        cc=cc,
        data=x,
        batch_size=batch_size,
        order=onp.COL_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )

In [124]:
x_plain.shape

(256, 1)

In [113]:
x_compat = make_compatible(x_plain, keys, debug=False)

(129, 129)


In [114]:
x_compat.decrypt(keys.secretKey, unpack_type="original")

array([0.10191404, 0.97993875, 0.45148197, 0.36976957, 0.6568808 ,
       0.62027872, 0.43156469, 0.01908877, 0.38121641, 0.60339516,
       0.39840004, 0.4345907 , 0.12768339, 0.19039507, 0.86352086,
       0.32197618, 0.25755605, 0.97280002, 0.5556078 , 0.45365804,
       0.25202191, 0.51811546, 0.07383761, 0.60484987, 0.0215516 ,
       0.76821607, 0.97783101, 0.70249242, 0.93789589, 0.35110214,
       0.22058818, 0.52879715, 0.42807442, 0.41568679, 0.06587133,
       0.641653  , 0.41658643, 0.71867204, 0.58494645, 0.26093724,
       0.90301448, 0.66585982, 0.33722743, 0.97222608, 0.44388479,
       0.45679152, 0.38358948, 0.38776934, 0.97463816, 0.08010671,
       0.00797783, 0.04303434, 0.38154832, 0.10034832, 0.16941798,
       0.51149672, 0.67959428, 0.21027768, 0.31421778, 0.22605699,
       0.81303614, 0.37431303, 0.5008834 , 0.33903325, 0.26390904,
       0.98220098, 0.87698478, 0.43132782, 0.26138288, 0.05261591,
       0.24349579, 0.36330938, 0.3619673 , 0.75642079, 0.17096

In [115]:
x_compat.shape, x_compat.original_shape

((256, 256), (129,))

## Seeing the shape of selection matrix

In [116]:
from openfhe_numpy.utils.matlib import next_power_of_two

In [119]:
selection_matrix_1 = np.random.rand(4, 12)

selection_matrix_1 = np.eye(129)

rows, cols = selection_matrix_1.shape
padded_rows = next_power_of_two(rows)
padded_cols = next_power_of_two(cols)
required_size = padded_rows * padded_cols

print(f"Original matrix: {rows} x {cols}")
print(f"Padded matrix: {padded_rows} x {padded_cols}")
print(f"Required size: {required_size}")
print(f"Your batch_size: {batch_size}")
print(f"Divisible: {batch_size % required_size == 0}")
print(f"Fits: {required_size <= batch_size}")


enc_selection_1 = onp.array(
        cc=cc,
        data=selection_matrix_1,
        batch_size=batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )

Original matrix: 129 x 129
Padded matrix: 256 x 256
Required size: 65536
Your batch_size: 65536
Divisible: True
Fits: True


In [118]:
selection_matrix_2 = np.random.rand(4, 4)


enc_selection_2 = onp.array(
        cc=cc,
        data=selection_matrix_2,
        batch_size=batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
enc_selection_2.shape, enc_selection_2.original_shape

((4, 4), (4, 4))